In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ai-tudy/finale/pipeline_detected_family_image

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ai-tudy/finale/pipeline_detected_family_image


In [2]:
import sys
sys.path.insert(0, './utils/')
from downloading_from_coco_2017 import download_image_from_coco_2017_for_model_has_human
from dataset import ConvDataset
from trainer import train_model, get_y_true_pred
from other_utils import get_model_resnet18, view_classification_report, load_model, save_json
from PIL import Image
from pathlib import Path
import pandas as pd
import json
import torch
import torch.nn as nn
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import os
from tqdm import tqdm
from copy import deepcopy
from sklearn.metrics import f1_score, classification_report



Скачивание изображений с COCO 2017

In [8]:
len_train_sample = 500
len_test_sample = int(len_train_sample * 0.3)
download_image_from_coco_2017_for_model_has_human(len_train_sample, len_test_sample)


Текстовые файлы аннотаций не найдены. Распаковываю локальный архив...
🎉 Аннотации успешно извлечены напрямую в data/instance/!
Скачано 50 фото
Скачано 100 фото
Скачано 150 фото
Скачано 200 фото
Скачано 250 фото
Скачано 300 фото
Скачано 350 фото
Скачано 400 фото
Скачано 450 фото
Итого скачано: 497
Скачано 50 фото
Скачано 100 фото
Скачано 150 фото
Скачано 200 фото
Скачано 250 фото
Скачано 300 фото
Скачано 350 фото
Скачано 400 фото
Скачано 450 фото
Скачано 500 фото
Итого скачано: 500
Скачано 50 фото
Скачано 100 фото
Итого скачано: 145
Скачано 50 фото
Скачано 100 фото
Скачано 150 фото
Итого скачано: 150


Основная работа по обучении модели.

In [3]:
root_dir_train = 'data/train/coco2017'
root_dir_val = 'data/val/coco2017'

dataset_train = ConvDataset(root_dir_train)
dataset_val = ConvDataset(root_dir_val)


In [4]:
train_loader = DataLoader(dataset_train, batch_size=32, shuffle=True, num_workers=4, pin_memory=True, prefetch_factor=4)
val_loader = DataLoader(dataset_val, batch_size=32, shuffle=False, num_workers=4, pin_memory=True, prefetch_factor=4)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_model_resnet18(num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()  # требует target: torch.long
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=6,
    eta_min=1e-7
)

In [6]:
path_save = 'models/has_human/best_model_has_human.pth'
history = train_model(model, train_loader, val_loader, path_save, criterion, optimizer, scheduler, device=device)



Epoch 1/15


Train:   0%|          | 0/32 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Test: 100%|██████████| 10/10 [00:02<00:00,  4.82it/s]


Train Loss: 0.3609 | Train Acc: 0.8305
Val   Loss: 0.2935 | Val   Acc: 0.8475
--------------------
--------------------

Epoch 2/15


Test: 100%|██████████| 10/10 [00:02<00:00,  4.15it/s]


++++++++++++++++++++++++++++
Early Stopping: 1 / 5 эпох без улучшений. (Лучший лосс: 0.2935150537450435, Текущий лосс: 0.3329711472078905)
++++++++++++++++++++++++++++
Train Loss: 0.0962 | Train Acc: 0.9759
Val   Loss: 0.3330 | Val   Acc: 0.8746
--------------------
--------------------

Epoch 3/15


...

++++++++++++++++++++++++++++
Early Stopping: 3 / 5 эпох без улучшений. (Лучший лосс: 0.2893859684467316, Текущий лосс: 0.3320564970121545)
++++++++++++++++++++++++++++
Train Loss: 0.0096 | Train Acc: 0.9980
Val   Loss: 0.3321 | Val   Acc: 0.8949
--------------------
--------------------

Epoch 13/15


Test: 100%|██████████| 10/10 [00:02<00:00,  4.78it/s]


++++++++++++++++++++++++++++
Early Stopping: 4 / 5 эпох без улучшений. (Лучший лосс: 0.2893859684467316, Текущий лосс: 0.38090771436691284)
++++++++++++++++++++++++++++
Train Loss: 0.0244 | Train Acc: 0.9930
Val   Loss: 0.3809 | Val   Acc: 0.8576
--------------------
--------------------

Epoch 14/15


Test: 100%|██████████| 10/10 [00:03<00:00,  2.99it/s]


++++++++++++++++++++++++++++
Early Stopping: 5 / 5 эпох без улучшений. (Лучший лосс: 0.2893859684467316, Текущий лосс: 0.4545446249388032)
++++++++++++++++++++++++++++
Train Loss: 0.0273 | Train Acc: 0.9920
Val   Loss: 0.4545 | Val   Acc: 0.8475
--------------------
Сработала рання остановка!


Тестирование на своих данных

In [7]:
root_dir_test = 'data/test/coco2017'
dataset_test = ConvDataset(root_dir_test)
test_loader = DataLoader(dataset_test, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
model = load_model("models/has_human/best_model_has_human.pth")

In [9]:
y_true, y_pred = get_y_true_pred(model, test_loader, str(device))

Test: 100%|██████████| 13/13 [00:18<00:00,  1.44s/it]


In [10]:
view_classification_report(y_true, y_pred, target_names=dataset_test.class_name_list)

               precision    recall  f1-score   support

    has_human       0.92      0.93      0.92        99
has_not_human       0.93      0.92      0.92       100

     accuracy                           0.92       199
    macro avg       0.92      0.92      0.92       199
 weighted avg       0.92      0.92      0.92       199



In [11]:
save_json(dataset_test.dict_class_label, 'labels/label_has_human.json')

JSON успешно сохранен по пути: labels/label_has_human.json
